# Required Imports

In [9]:
import os
import pandas as pd
import re
import matplotlib.pyplot as plt
from io import BytesIO
from scipy.io import loadmat  # this is the SciPy module that loads mat-files

import torch
from torch_geometric.data import Data

# Imports PIL module
from PIL import Image

In [10]:
# Configuration Settings
n_packets = 100 # Number of packets within each packet group per ID, use even numbers. This defines the number of packet groups per ID (total_packets/n_packets)

# Dataset Generation

## Load IQR Amplitude values 

In [11]:
# Define file ID
def extract_id_from_filename(filename):
    """ Extract the ID from the filename, including the number and the trailing 'A' or 'B'. """
    match = re.search(r'(\d+[mf][AB])\.', filename) # e.g., 00001mA
    return match.group(1) if match else None

# Load amplitudes from CSV files
def load_amplitudes(directory):
    # Initialize a dictionary to hold amplitudes
    amplitudes = {}

    for file_name in os.listdir(directory):
        if file_name.endswith('.csv'):
            file_path = os.path.join(directory, file_name)

            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path)

            # Extract ID from the file name
            file_id = extract_id_from_filename(file_name)
            if file_id:
                # Collect amplitudes per ID[mf][AB] and store them in a dictionary
                amplitudes[file_id] = df

    return amplitudes

# Folder to save the processed files
output_folder = './output/'

# Directory containing the CSV files to process
input_directory = os.path.join(output_folder,'CSV','sanitized_amplitudes', 'IQR_amplitudes')

# Process and save the files
sanitized_amplitudes = load_amplitudes(input_directory)

In [12]:

# This dict will contain the packet groups for each ID
split_sanitized_amplitudes = {}

# Splitting the sanitized amplitudes into groups of 'n_packets' per ID
for ID, amplitude_values in sanitized_amplitudes.items():
    split_sanitized_amplitudes[ID] = []
    # Calculate the number of full groups that can be made
    num_full_groups = len(amplitude_values) // n_packets

    for i in range(num_full_groups):
        # Calculate the start index for each group
        start_index = i * n_packets
        # Calculate the end index for each group
        end_index = start_index + n_packets
        # Extract the group from amplitude_values
        split_amplitude_group = amplitude_values[start_index:end_index]
        # Append the group to the corresponding ID's list
        split_sanitized_amplitudes[ID].append(split_amplitude_group)

    # Handle any remaining elements that didn't form a full group
    if len(amplitude_values) % n_packets != 0:
        # Add the remaining elements as a separate group
        split_sanitized_amplitudes[ID].append(amplitude_values[num_full_groups * n_packets:])

## Split data into Train, Validation, and Test sets

In [13]:
import random

# Set the seed to make the split reproducible across experiments
random.seed(42)

# Group packets by base identity (e.g., from '00001mA' to '00001m')
groups = {}
for key, packets in split_sanitized_amplitudes.items():
    base_id = key[:-1] # Get everything except the last letter (which is 'A' or 'B')
    session = key[-1]  # 'A' or 'B'
    
    if base_id not in groups:
        groups[base_id] = {'A': [], 'B': []}
    
    # Add the packets to their respective session
    groups[base_id][session].extend(packets)

# Extract all 81 unique identities and shuffle them
all_identities = list(groups.keys())
random.shuffle(all_identities)

# Re-ID split by IDENTITIES: 40 IDs for Training, 41 IDs for Test
train_ids = all_identities[:40]
test_ids = all_identities[40:]

print(f"Subjects in Training Set: {len(train_ids)}")
print(f"Subjects in Test Set: {len(test_ids)}")

Subjects in Training Set: 40
Subjects in Test Set: 41


## Generate Probes and Gallery for each set

In [14]:
train_probe = {}
train_gallery = {}
test_probe = {}
test_gallery = {}
val_probe = {}   # Initialized empty because in a pure Re-ID setup, validation 
val_gallery = {} # is either done via cross-fold on the train set, or the original set isn't used.

# 1. Training Set Construction (Merge Session A and Session B)
for base_id in train_ids:
    # Combine heatmaps from Session A and Session B to enforce invariance
    all_packets = groups[base_id]['A'] + groups[base_id]['B']
    
    # Split the total packets in half (half for probe, half for gallery) 
    # so the Siamese Network has pairs to learn the loss
    half = len(all_packets) // 2
    train_probe[base_id] = all_packets[:half]
    train_gallery[base_id] = all_packets[half:]

# 2. Test Set Construction (Gallery = Session A, Probe = Session B)
for base_id in test_ids:
    # Gallery: database of known identities (Session A only)
    test_gallery[base_id] = groups[base_id]['A']
    
    # Probe: queries to search for (Session B only)
    test_probe[base_id] = groups[base_id]['B']
    
print(f"Generated training data for {len(train_probe)} identities.")
print(f"Generated test data (Gallery A / Probe B) for {len(test_probe)} identities.")

Generated training data for 40 identities.
Generated test data (Gallery A / Probe B) for 41 identities.


## Generate Magnitude Heatmaps



In [15]:
import torch
from torch_geometric.data import Data
import os

def save_graph_data(data_matrix, subject_id, output_path):
    # 1. Nodes (x): Transpose from (100 packets, 52 subcarriers) to (52 nodes, 100 temporal features)
    x = torch.tensor(data_matrix.T, dtype=torch.float32)
    
    # 2. Edges (edge_index): Create a FULLY CONNECTED graph (Complete Graph)
    num_nodes = x.shape[0] # 52
    
    # Generate all possible pairs of indices (0 to 51) using meshgrid
    row_idx, col_idx = torch.meshgrid(torch.arange(num_nodes), torch.arange(num_nodes), indexing='ij')
    
    # Flatten the grids to create the source and target edge lists
    sources = row_idx.flatten()
    targets = col_idx.flatten()
    
    # Create the edge_index tensor directly
    edge_index = torch.stack([sources, targets], dim=0).to(torch.long)
    
    # 3. Create PyG Graph Data
    graph_data = Data(x=x, edge_index=edge_index, subject_id=subject_id)
    
    # 4. Save to PyTorch tensor format
    torch.save(graph_data, output_path)

##################### TRAIN GRAPHS ##########################

# Generate and save graph data for train probes
if train_probe:
  # Folder to save the train probes
  output_path = os.path.join(output_folder, 'GraphData', 'D1', 'Probe')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the train_probe dictionary
  for ID, group_list in train_probe.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_packets in enumerate(group_list):
          # Convert the group packets to a numpy array and drop the timestamp column
          group_values = group_packets.drop(columns='timestamp').to_numpy()
          
          # Define the filename using the probe ID and group index (.pt instead of .png)
          filename = f"{ID}_{group_idx}.pt"
          
          # Save the graph data for the current group
          save_graph_data(group_values, ID, os.path.join(output_path, filename))

# Generate and save graph data for train gallery
if train_gallery:
  # Folder to save the train gallery
  output_path = os.path.join(output_folder, 'GraphData', 'D1', 'Gallery')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the train_gallery dictionary
  for ID, group_list in train_gallery.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_packets in enumerate(group_list):
          # Convert the group packets to a numpy array and drop the timestamp column
          group_values = group_packets.drop(columns='timestamp').to_numpy()
          
          # Define the filename using the probe ID and group index (.pt instead of .png)
          filename = f"{ID}_{group_idx}.pt"
          
          # Save the graph data for the current group
          save_graph_data(group_values, ID, os.path.join(output_path, filename))

##################### VAL GRAPHS ############################

# Generate and save graph data for val probes
if val_probe:
  # Folder to save the val probes
  output_path = os.path.join(output_folder, 'GraphData', 'D2v', 'Probe')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the val_probe dictionary
  for ID, group_list in val_probe.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_packets in enumerate(group_list):
          # Convert the group packets to a numpy array and drop the timestamp column
          group_values = group_packets.drop(columns='timestamp').to_numpy()
          
          # Define the filename using the probe ID and group index
          filename = f"{ID}_{group_idx}.pt"
          
          # Save the graph data for the current group
          save_graph_data(group_values, ID, os.path.join(output_path, filename))

# Generate and save graph data for val gallery
if val_gallery:
  # Folder to save the val gallery
  output_path = os.path.join(output_folder, 'GraphData', 'D2v', 'Gallery')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the val_gallery dictionary
  for ID, group_list in val_gallery.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_packets in enumerate(group_list):
          # Convert the group packets to a numpy array and drop the timestamp column
          group_values = group_packets.drop(columns='timestamp').to_numpy()
          
          # Define the filename using the probe ID and group index
          filename = f"{ID}_{group_idx}.pt"
          
          # Save the graph data for the current group
          save_graph_data(group_values, ID, os.path.join(output_path, filename))

##################### TEST GRAPHS ##########################

# Generate and save graph data for test probes
if test_probe:
  # Folder to save the test probes
  output_path = os.path.join(output_folder, 'GraphData', 'D2t', 'Probe')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the test_probe dictionary
  for ID, group_list in test_probe.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_packets in enumerate(group_list):
          # Convert the group packets to a numpy array and drop the timestamp column
          group_values = group_packets.drop(columns='timestamp').to_numpy()
          
          # Define the filename using the probe ID and group index
          filename = f"{ID}_{group_idx}.pt"
          
          # Save the graph data for the current group
          save_graph_data(group_values, ID, os.path.join(output_path, filename))

# Generate and save graph data for test gallery
if test_gallery:
  # Folder to save the test gallery
  output_path = os.path.join(output_folder, 'GraphData', 'D2t', 'Gallery')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the test_gallery dictionary
  for ID, group_list in test_gallery.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_packets in enumerate(group_list):
          # Convert the group packets to a numpy array and drop the timestamp column
          group_values = group_packets.drop(columns='timestamp').to_numpy()
          
          # Define the filename using the probe ID and group index
          filename = f"{ID}_{group_idx}.pt"
          
          # Save the graph data for the current group
          save_graph_data(group_values, ID, os.path.join(output_path, filename))

#!zip -r /content/CSV.zip /content/output/CSV
#!zip -r /content/GraphData.zip /content/output/GraphData

In [16]:
import torch
from torch_geometric.data import Data
import os

# Definiamo la cartella invece del singolo file
probe_dir = os.path.join(output_folder, 'GraphData', 'D1', 'Probe')

if os.path.exists(probe_dir):
    # Peschiamo tutti i file .pt nella cartella
    pt_files = [f for f in os.listdir(probe_dir) if f.endswith('.pt')]
    
    if pt_files:
        # Prendiamo il primissimo file della lista
        sample_file = os.path.join(probe_dir, pt_files[0])
        print(f"File trovato con successo: {sample_file}\n")
        
        # 1. Carica il grafo
        graph_data = torch.load(sample_file, weights_only=False)
        
        # 2. Print general overview
        print("=== Object Overview ===")
        print(graph_data)
        
        # 3. Check exact dimensions (Sanity Check)
        print("\n=== Tensor Details ===")
        print(f"Node shape (x): {graph_data.x.shape}") 
        print(f"Edge shape (edge_index): {graph_data.edge_index.shape}")
        print(f"Saved Subject ID: {graph_data.subject_id}")
        
        # 4. Useful PyTorch Geometric native methods
        print("\n=== Structural Properties ===")
        print(f"Total number of nodes: {graph_data.num_nodes}")
        print(f"Total number of edges: {graph_data.num_edges}")
        print(f"Are there isolated nodes? {graph_data.has_isolated_nodes()}")
    else:
        print("La cartella esiste, ma è vuota. Nessun file .pt trovato al suo interno.")
else:
    print(f"Cartella non trovata: {probe_dir}. Assicurati che lo script di generazione abbia girato correttamente.")

File trovato con successo: ./output/GraphData/D1/Probe/00038m_2.pt

=== Object Overview ===
Data(x=[52, 100], edge_index=[2, 2704], subject_id='00038m')

=== Tensor Details ===
Node shape (x): torch.Size([52, 100])
Edge shape (edge_index): torch.Size([2, 2704])
Saved Subject ID: 00038m

=== Structural Properties ===
Total number of nodes: 52
Total number of edges: 2704
Are there isolated nodes? False
